# Task 5 — Train / Validation / Test Split

This notebook demonstrates the reusable splitting utility in `scripts/modeling_split.py`.

Core rules implemented:
- Test is held out **first**.
- Test remains untouched until final evaluation.
- Remaining data is split into Train and Validation.
- A fixed `random_state` makes the split reproducible.
- Target distributions are compared across all three splits.
- The actual target and final features are intentionally supplied by Tasks 1–4.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from scripts.modeling_split import (
    split_train_validation_test,
    split_size_summary,
    compare_target_distributions,
)


## Demonstration

The demo uses a small synthetic regression dataset only to verify Task 5 infrastructure. When Tasks 1–4 provide the project-specific `X` and `y`, the same functions are used without changing the split logic.


In [ ]:
rng = np.random.default_rng(42)
n = 1000

X = pd.DataFrame({
    'building_size': rng.normal(100, 25, n),
    'rooms_count': rng.integers(1, 5, n),
    'city_slug': rng.choice(['tehran', 'karaj', 'shiraz'], n),
})

y = pd.Series(
    rng.lognormal(mean=22.0, sigma=0.7, size=n),
    name='target',
)

splits = split_train_validation_test(
    X,
    y,
    test_size=0.15,
    validation_size=0.15,
    random_state=42,
)


In [ ]:
split_size_summary(splits)


In [ ]:
compare_target_distributions(splits)


## Integration with the real modeling dataset

After Tasks 1–4 define the target and final feature set, use:

```python
X = modeling_df[final_feature_columns]
y = modeling_df[target_column]

splits = split_train_validation_test(
    X,
    y,
    test_size=0.15,
    validation_size=0.15,
    random_state=42,
)

X_train = splits.X_train
X_validation = splits.X_validation
X_test = splits.X_test
y_train = splits.y_train
y_validation = splits.y_validation
y_test = splits.y_test
```

Only `X_train` / `y_train` should be used to **fit** preprocessing or models. `Validation` is used for model comparison/tuning, and `Test` is reserved for the final evaluation.
